# Idea 1 pilot: layer sweep and learned layer mixture (CPU)
Reads the cached 26-layer AraBEST-RQ features of the 700 frozen pilot clips from Drive and trains the
Idea 1 heads on CPU. No GPU, encoder or audio is used. Method: `docs/layer_mixture.md`;
settings: `configs/layer_mixture.json`.

**Code source.** The script was not pushed when this ran, so Cell 1 uploads
`Arabic-Dialect-Identification-idea1.zip`: the repository at base commit `31013e56` plus the uncommitted
Idea 1 files, with `.git` included. The run's `run_record.json` therefore names that commit and
`dirty: true`. The files that matter, as run on 2026-11-19:
- `src/experiments/layer_mixture.py` SHA256 `9d55adc17586cb1a6ec9df6462f8662cf4df19f8e945c9d818e40c1c2d8e79ac`
- `configs/layer_mixture.json` SHA256 `5f27ea9ce4c41c22c25df02cd256415c0e69e21597efc12651a29b6b40301a1a`

These are the cells used for `output/runs/layer_mixture/run1`. The last cell packages that run
folder for sharing.

In [ ]:
# Cell 1: mount Drive, upload and unpack the code, check the feature cache
from google.colab import drive, files
import zipfile
from pathlib import Path
drive.mount('/content/drive')
uploaded = files.upload()  # choose Arabic-Dialect-Identification-idea1.zip
REPO = Path('/content/Arabic-Dialect-Identification')
zipfile.ZipFile('Arabic-Dialect-Identification-idea1.zip').extractall(REPO)
OUTPUT_ROOT = '/content/drive/MyDrive/Arabic-Dialect-Identification-data/output'
CACHE = Path(OUTPUT_ROOT) / 'features/arabest-rq-300m-6k/dff453e2b5f7/e4b3c287854e40f9'
print('cache found:', CACHE.is_dir(), '| cached clips:', sum(1 for _ in open(CACHE / 'index.jsonl')))

In [ ]:
# Cell 2: run the pilot (about 35 minutes on a Colab CPU, one thread)
!pip -q install soundfile
RUN_DIR = f'{OUTPUT_ROOT}/runs/layer_mixture/run1'
!cd {REPO} && python src/experiments/layer_mixture.py --output-root "{OUTPUT_ROOT}" --output-dir "{RUN_DIR}"

In [ ]:
# Cell 3 (optional): all 26 single-layer probes
import pandas as pd
pd.read_csv(f'{RUN_DIR}/layer_sweep.csv')

## Package the run folder for a teammate
Self-contained: works in a fresh session after mounting Drive. It prints the code and config
hashes the run recorded, checks them against the files listed above, prints the shared evaluator's
accuracy for every headline arm and seed, and writes `run1_share.zip` next to the run folder.

In [ ]:
import hashlib, json, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data/output/runs/layer_mixture/run1')
record = json.loads((RUN_DIR / 'run_record.json').read_text())
print('commit', record['code']['commit'], 'dirty', record['code']['dirty'])
print('script matches:', record['code']['script_sha256'] == '9d55adc17586cb1a6ec9df6462f8662cf4df19f8e945c9d818e40c1c2d8e79ac')
print('config matches:', record['code']['config_sha256'] == '5f27ea9ce4c41c22c25df02cd256415c0e69e21597efc12651a29b6b40301a1a')
for arm in ['b0_final', 'b0_final_raw', 'selected_layer', 'mixture_all', 'mixture_middle']:
    metrics = [json.loads((RUN_DIR / 'arms' / arm / f'seed_{s}' / 'evaluation/metrics.json').read_text())
               for s in (490, 491, 492)]
    print(f"{arm:15s} evaluate.py accuracy {[round(m['accuracy'], 4) for m in metrics]}")
archive = shutil.make_archive(str(RUN_DIR.parent / 'run1_share'), 'zip', RUN_DIR)
print(archive, round(Path(archive).stat().st_size / 1e6, 2), 'MB',
      hashlib.sha256(Path(archive).read_bytes()).hexdigest())